# Inference optimization

**Track:** Portfolio. **Level:** Practitioner. **CPU:** this notebook. **GPU:** one cell, skipped unless `RUN_GPU=1`.

## Goals

- Compare per-example latency of batch size 1 with a larger batch on a small transformer block.
- Quantize the linear layers of an MLP to int8 and measure the numerical error.
- Leave a CUDA timing cell in the notebook and make CPU CI skip it.

## Sources and license

Original notebook. Dynamic quantization uses PyTorch (BSD-style license). The reading list is FlashAttention (Dao et al.), GPTQ (Frantar et al.), QLoRA (Dettmers et al.), speculative decoding (Leviathan et al.), and PagedAttention (Kwon et al.). None of those kernels are reimplemented here. At this batch size a kernel can be slower after quantization because the dispatch costs more than the arithmetic saves. The notebook reports that instead of assuming a speedup.

## Concepts

Throughput is examples per second. Latency is time per call. Batching improves throughput when the extra examples share the kernel launch. It does not improve the latency of a single request unless you wait for a batch to fill. Quantization reduces bytes moved per multiply. The check that matters next to the timer is the max absolute difference on the output, because a faster wrong answer is a regression.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'matplotlib', 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from labs._shared.seed import seed_everything

seed_everything(0)
torch.set_num_threads(2)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


## Guided exercise: batching

The block is one multi-head attention layer and a feed-forward layer, batch-first, width 64, sequence 32. The timer warms up, then measures a call. The quantity compared across batch sizes is seconds per example.


In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d: int = 64, nhead: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, nhead, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.norm1(x + h)
        return self.norm2(x + self.ff(x))

block = TinyBlock().eval()

def bench(module, batch: int, seq: int = 32, d: int = 64, repeats: int = 8) -> float:
    x = torch.randn(batch, seq, d)
    with torch.no_grad():
        for _ in range(2):
            module(x)
        start = time.perf_counter()
        for _ in range(repeats):
            module(x)
        elapsed = time.perf_counter() - start
    return elapsed / repeats

rows = []
for batch in (1, 8, 32):
    elapsed = bench(block, batch)
    rows.append({"batch": batch, "seconds": elapsed, "seconds_per_example": elapsed / batch})
batch_df = __import__("pandas").DataFrame(rows)
print(batch_df.round(5).to_string(index=False))
assert batch_df.loc[batch_df["batch"] == 32, "seconds_per_example"].item() < batch_df.loc[batch_df["batch"] == 1, "seconds_per_example"].item()


## Guided exercise: dynamic int8 quantization

Dynamic quantization converts `nn.Linear` weights to int8 and quantizes activations at runtime. The MLP is wide enough that the output difference is visible and still small relative to the scale of a random network. Error is max absolute difference on a fixed input, not a language-model benchmark.


In [ ]:
class MLP(nn.Module):
    def __init__(self, d=256, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d),
        )

    def forward(self, x):
        return self.net(x)

mlp = MLP().eval()
try:
    from torch.ao.quantization import quantize_dynamic
except ImportError:
    from torch.quantization import quantize_dynamic
qmlp = quantize_dynamic(mlp, {nn.Linear}, dtype=torch.qint8)
qmlp.eval()

probe = torch.randn(32, 256)
with torch.no_grad():
    fp = mlp(probe)
    qi = qmlp(probe)
max_abs = float((fp - qi).abs().max())
fp_time = bench_mlp = None

def bench_vec(module, repeats=10):
    x = torch.randn(64, 256)
    with torch.no_grad():
        for _ in range(2):
            module(x)
        start = time.perf_counter()
        for _ in range(repeats):
            module(x)
        return (time.perf_counter() - start) / repeats

fp_time = bench_vec(mlp)
qi_time = bench_vec(qmlp)
print(f"max abs diff {max_abs:.4f}  fp32 {fp_time:.4f}s  int8 {qi_time:.4f}s")
assert max_abs < 1.5
assert np.isfinite(fp_time) and np.isfinite(qi_time)


## Your turn

The GPU cell is part of the lab and is not part of CPU CI. It runs only when `RUN_GPU=1` and CUDA is visible. On this machine it should print the skip line.

## Solution


In [ ]:
import os
run_gpu = os.environ.get("RUN_GPU", "0") == "1" and torch.cuda.is_available()
if not run_gpu:
    print("SKIP GPU cell: set RUN_GPU=1 on a CUDA machine. CPU CI does not run this cell.")
else:
    device = torch.device("cuda")
    gpu_block = TinyBlock().to(device).eval()
    x = torch.randn(32, 32, 64, device=device)
    for _ in range(5):
        gpu_block(x)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(20):
        gpu_block(x)
    torch.cuda.synchronize()
    print("gpu seconds", (time.perf_counter() - start) / 20)


## Beyond the course

Plot seconds per example against batch size, and state whether int8 was faster or slower at this width. The research point is the crossover: quantization wins when weight traffic dominates launch overhead. Record the direction you observed. The assertion checks the error budget, not a required speedup.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(batch_df["batch"], batch_df["seconds_per_example"], marker="o")
ax.set_xlabel("batch")
ax.set_ylabel("seconds per example")
ax.set_title("Attention block, CPU")
plt.tight_layout()
plt.show()
print("int8 faster than fp32" if qi_time < fp_time else "int8 slower than fp32 at this width")
assert max_abs < 1.5
print("LAB OK")


## Research questions

1. Per-example time fell with batch size. Did the single-example latency fall? What does a serving system do with a request that cannot wait for 32 neighbors?
2. If int8 was slower here, which tensor in the MLP would you expect to dominate once the hidden size grows by 10x?
3. FlashAttention changes the memory traffic of the attention scores. Why is that a different bottleneck from quantizing `nn.Linear`?

## Further reading

- Jacob et al., *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference* (2018).
- Dettmers et al., *QLoRA* (2023).
- Frantar et al., *GPTQ* (2022).
- Dao et al., *FlashAttention* (2022).
- Leviathan, Kalman, and Matias, *Fast Inference from Transformers via Speculative Decoding* (2023).
- Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (2023).
